> **Historical notebook: Wolff reference visualisation (L=32).**
> This notebook belongs to the earlier Wolff / supervised Flow Matching stage; it does not implement the current data-free SPS method. It visualises reference configurations and plaquette winding numbers.
> The original code is retained, with saved outputs removed for publication. A clean clone does not contain the required `data/xy_L32_T0.89_10k.pt`, `data/xy_L32_T0.5.pt`, or `data/xy_L32_T1.5.pt` tensors. Existing experiment copies require those original files; newly generated data are new runs.
> Read the [historical baseline guide](../docs/public/legacy.md) for the reference figures, interpretation limits, and reproducibility requirements.


# XY 模型数据验证与涡旋可视化

本笔记本用于：
1. 加载 Wolff 采样生成的 XY 模型构型数据
2. 从双通道 $(\cos\theta, \sin\theta)$ 张量还原角度矩阵
3. 计算 Plaquette 涡旋荷 (Vorticity)
4. 高质量可视化：角度热力图 + 自旋矢量场 + 涡旋标记

说明：本笔记用于认识自旋和涡旋。单张图不能证明涡旋解束缚；净涡旋荷为零是周期边界下的代数约束。图片统一保存到 `figures/`。


In [ ]:
from pathlib import Path

# Find the project from either the repository root or notebooks/.
PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'src' / 'model_dit.py').is_file() and (p / 'data').is_dir()
)
FIGURE_DIR = PROJECT_ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

import os
import numpy as np
import torch
import matplotlib.pyplot as plt

%matplotlib inline

L = 32  # 晶格尺寸

## 1. 加载数据

读取 `data/xy_L32_T0.89_10k.pt`，张量形状为 $(10000, 2, 32, 32)$，
通道 0 = $\cos\theta$，通道 1 = $\sin\theta$。

In [ ]:
data_path = PROJECT_ROOT / 'data' / 'xy_L32_T0.89_10k.pt'

# 兼容不同 PyTorch 版本的 weights_only 参数
try:
    tensor = torch.load(data_path, weights_only=False)
except TypeError:
    tensor = torch.load(data_path)

print(f'Tensor shape: {tensor.shape}')
print(f'Tensor dtype: {tensor.dtype}')
print('Channels: 0 = cos θ, 1 = sin θ')

# 随机抽取一个样本 (可修改 seed 获取不同样本)
np.random.seed(42)
sample_idx = np.random.randint(0, tensor.shape[0])
sample = tensor[sample_idx]  # shape: (2, 32, 32)
print(f'\nRandomly selected sample #{sample_idx}')
print(f'Sample shape: {sample.shape}')

## 2. 角度还原

利用 $\theta = \operatorname{atan2}(\sin\theta, \cos\theta)$ 将双通道张量还原为角度矩阵，
输出范围为 $[-\pi, \pi)$。

In [ ]:
cos_theta = sample[0].numpy()  # (32, 32)
sin_theta = sample[1].numpy()  # (32, 32)

theta = np.arctan2(sin_theta, cos_theta)  # [-π, π)

print(f'Angle matrix shape: {theta.shape}')
print(f'Angle range: [{theta.min():.4f}, {theta.max():.4f}]')
print(f'Theoretical range: [-π, π) = [-{np.pi:.4f}, {np.pi:.4f})')

## 3. 拓扑涡旋 (Vorticity) 计算

遍历所有最小闭合回路 (Plaquette)：
$$(i,j) \to (i{+}1,j) \to (i{+}1,j{+}1) \to (i,j{+}1) \to (i,j)$$

- **相位差截断**：相邻自旋角度差 $\Delta\theta$ 通过 wrap 映射到 $[-\pi, \pi)$
- **涡旋荷**：4 条边 wrapped_diff 之和 $\div 2\pi$，四舍五入取整
- $+1$ = 正涡旋，$-1$ = 反涡旋

In [ ]:
def calculate_vorticity(theta):
    """Compute the Plaquette vorticity of the 2D XY model.

    Args:
        theta: (L, L) angle matrix
    Returns:
        vorticity: (L, L) integer array, +1=vortex, -1=anti-vortex, 0=none
    """
    L = theta.shape[0]
    vorticity = np.zeros((L, L), dtype=int)

    for i in range(L):
        for j in range(L):
            ip1 = (i + 1) % L  # 周期性边界 (PBC)
            jp1 = (j + 1) % L

            # 回路: (i,j) → (i+1,j) → (i+1,j+1) → (i,j+1) → (i,j)
            diffs = [
                theta[ip1, j]    - theta[i, j],       # (i,j) → (i+1,j)
                theta[ip1, jp1]  - theta[ip1, j],     # (i+1,j) → (i+1,j+1)
                theta[i, jp1]   - theta[ip1, jp1],   # (i+1,j+1) → (i,j+1)
                theta[i, j]     - theta[i, jp1],      # (i,j+1) → (i,j)
            ]

            total = 0.0
            for d in diffs:
                # wrap 到 [-π, π)
                d_wrapped = (d + np.pi) % (2 * np.pi) - np.pi
                total += d_wrapped

            vorticity[i, j] = int(round(total / (2 * np.pi)))

    return vorticity


# 计算涡旋荷
vorticity = calculate_vorticity(theta)

n_pos = int(np.sum(vorticity == 1))
n_neg = int(np.sum(vorticity == -1))
print(f'Vortices (+1): {n_pos}')
print(f'Anti-vortices (-1): {n_neg}')
print(f'Net vorticity: {vorticity.sum()} (PBC identity; not a quality score)')

## 4. 可视化

- **背景热力图**：角度 $\theta$（HSV 色轮天然适合表示周期量）
- **矢量场**：`plt.quiver` 绘制 $(\cos\theta, \sin\theta)$
- **涡旋标记**：红色圆点 = $+1$，蓝色圆点 = $-1$

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

# --- 背景: 角度热力图 (HSV 色轮天然适合表示角度) ---
im = ax.imshow(theta, cmap='hsv', origin='lower',
               extent=[-0.5, L - 0.5, -0.5, L - 0.5],
               vmin=-np.pi, vmax=np.pi, interpolation='nearest')

# --- 矢量场: 自旋方向 (cos θ, sin θ) ---
x = np.arange(L)
y = np.arange(L)
X, Y = np.meshgrid(x, y)
U = np.cos(theta)
V = np.sin(theta)
ax.quiver(X, Y, U, V, color='black', scale=30, width=0.004,
          headwidth=3, headlength=4, pivot='middle')

# --- 涡旋标记 (位于 Plaquette 中心) ---
pos = np.argwhere(vorticity == 1)
neg = np.argwhere(vorticity == -1)

if len(pos) > 0:
    ax.plot(pos[:, 1] + 0.5, pos[:, 0] + 0.5, 'o',
            color='red', markersize=9, markeredgecolor='darkred',
            markeredgewidth=1.2, label=f'Vortex (+1) (n={n_pos})', zorder=5)
if len(neg) > 0:
    ax.plot(neg[:, 1] + 0.5, neg[:, 0] + 0.5, 'o',
            color='blue', markersize=9, markeredgecolor='darkblue',
            markeredgewidth=1.2, label=f'Anti-vortex (-1) (n={n_neg})', zorder=5)

# --- 装饰 ---
ax.set_xlim(-0.5, L - 0.5)
ax.set_ylim(-0.5, L - 0.5)
ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('y', fontsize=13)
ax.set_title(f'XY Model Spin Configuration (L={L}, T≈0.89)\n'
             f'Sample #{sample_idx} | +1: {n_pos}  -1: {n_neg}', fontsize=14)
ax.legend(fontsize=11, loc='upper right', framealpha=0.9)

cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('θ (rad)', fontsize=12)

plt.tight_layout()
plt.show()

## 5. 物理现象验证：涡旋对解绑 (Vortex Unbinding) 随温度的演化
对比三个典型温度下生成的物理构型：
- $T=0.5 < T_c$ (低温)：准长程有序，涡旋极少且紧密配对 (Bound Pairs)。
- $T \approx 0.89$ (临界点)：吸引势被打破，涡旋对开始游离解绑 (Unbinding)。
- $T=1.5 > T_c$ (高温)：彻底无序，出现大量随机分布的涡旋等离子体 (Vortex Plasma)。


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 读取三个不同温度的数据张量 (各取第 0 个样本)
t_low = torch.load(PROJECT_ROOT / 'data' / 'xy_L32_T0.5.pt', weights_only=False)[0].numpy()
t_crit = torch.load(PROJECT_ROOT / 'data' / 'xy_L32_T0.89_10k.pt', weights_only=False)[0].numpy()
t_high = torch.load(PROJECT_ROOT / 'data' / 'xy_L32_T1.5.pt', weights_only=False)[0].numpy()

data_samples = {
    r"Low T ($T=0.5$): Bound Pairs": t_low,
    r"Critical T ($T \approx 0.89$): Unbinding": t_crit,
    r"High T ($T=1.5$): Vortex Plasma": t_high
}

plt.rcParams.update({'font.family': 'serif', 'font.size': 12})
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (title, sample) in zip(axes, data_samples.items()):
    # 1. 还原角度
    cos_theta, sin_theta = sample[0], sample[1]
    theta = np.arctan2(sin_theta, cos_theta)

    # 2. 计算涡旋 (依赖前面定义的 calculate_vorticity 函数)
    vorticity = calculate_vorticity(theta)
    pos_vortices = np.argwhere(vorticity == 1)
    neg_vortices = np.argwhere(vorticity == -1)

    # 3. 绘制热力图与矢量场
    X, Y = np.meshgrid(np.arange(32), np.arange(32))
    im = ax.imshow(theta, cmap='hsv', origin='lower', vmin=-np.pi, vmax=np.pi, alpha=0.8)
    ax.quiver(X, Y, cos_theta, sin_theta, color='black', pivot='middle', scale=35)

    # 4. 叠加红蓝涡旋点
    ax.scatter(pos_vortices[:, 1], pos_vortices[:, 0], c='red', s=60, edgecolors='black', label='+1 Vortex')
    ax.scatter(neg_vortices[:, 1], neg_vortices[:, 0], c='blue', s=60, edgecolors='black', label='-1 Anti-vortex')

    ax.set_title(f"{title}\n(+1: {len(pos_vortices)}, -1: {len(neg_vortices)})")
    ax.set_xticks([])
    ax.set_yticks([])

# 统一的图例
axes[0].legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'vortex_unbinding_transition.png', dpi=300)
plt.show()
